In [ ]:
# import libraries
import polars as pl
import plotly.express as px
from pathlib import Path

# uploaded dataset & quick check of column types
file_path = Path("Traffic_Volumes_-5908226106492997850.csv")
df = pl.read_csv(file_path)
print(df.head())
print(df.describe())
print(df.null_count())

# clean edges and convert text to uppercase & conver str to datetime formate
df = df.with_columns([
    pl.col([
        "CARTO_CLASS", "FLOW_DIRECTION",
    ]).str.strip_chars().str.to_uppercase(),
    pl.col("SOURCE_DATE")
    .str.strptime(
    pl.Datetime, "%m/%d/%Y %I:%M:%S %p", strict=False
    )

])

# created dictionary for column type casting and apply it
col_types = {
     "AADT" : pl.Int32,
    "AADT_YEAR" : pl.Int16,
    "LANES" : pl.Int8,
    "SPEED_LIMIT" : pl.Int8
}
df = df.with_columns([
    pl.col(col).cast(dtype, strict=False).alias(col)
    for col, dtype in col_types.items()
])

# made dropping list column and drop it
columns_to_drop = [
    "REGIONAL_ROAD",
    "SPEED_85TH_KM",
    "MEAN_SPEED_KM",
    "DBT",
    "PERCENT_BIKES",
    "TRUCK_ACCESS"
]
df = df.drop(columns_to_drop)

# dropped missing values
df = df.drop_nulls()

# created new row_id
df = df.with_row_index("row_id", offset=1)

# upload clean dataset version
df.write_csv("cleaned_traffic.csv")
print(df.head())
print(df.describe())
print(df.null_count())


# aggregation
df_carto_class = (
    df.group_by(["CARTO_CLASS", "FLOW_DIRECTION"])
     .agg([
      pl.mean("AADT").alias("avg_aadt"),
      pl.mean("SPEED_LIMIT").alias("avg_speed_limit")
  ])
    .sort("avg_aadt", descending=True)
)


df_lanes = (
    df.group_by(["LANES", "CARTO_CLASS"])
     .agg([
      pl.mean("AADT").alias("avg_aadt"),
     pl.mean("SPEED_LIMIT").alias("avg_speed_limit")
  ])
    .sort("avg_aadt", descending=True)
)

df_aadt_year = (
    df.group_by(["AADT_YEAR", "CARTO_CLASS"])
     .agg([
     pl.mean("AADT").alias("avg_aadt"),
     pl.median("AADT").alias("aadt_median")
  ])
    .sort("AADT_YEAR", descending=True)
)

# conver to pandas for Plotly
carto_class_pd = df_carto_class.to_pandas()
lanes_pd = df_lanes.to_pandas()
aadt_year_pd = df_aadt_year.to_pandas()

# visualization
fig1 = px.bar(
    carto_class_pd,
    x="CARTO_CLASS",
    y="avg_aadt",
    color="FLOW_DIRECTION",
    barmode="group",
    text_auto=".3s"  ,
    hover_data=["avg_speed_limit"],
    title="Distribution of Traffic Volume by Road Class & Flow Direction"
)
fig1.show()

fig2 = px.scatter(
    lanes_pd,
    x="LANES",
    y="avg_aadt",
    color="CARTO_CLASS",
    size="avg_aadt",
    trendline="ols",
    title="Correlation Between Lane Capacity & AADT"
)
fig2.show()

fig3 = px.line(
    aadt_year_pd,
    x="AADT_YEAR",
    y=["avg_aadt",
     "aadt_median"],
    color="CARTO_CLASS",
    title="Evolution of Traffic Volumes by Road Class : Mean vs Median (1990 - 2024)"
)
fig3.show()